In [1]:
import os
import sys
import glob
import json
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, round, sum as spark_sum, count as spark_count

print("=======================================================")
print("      MULTI-SYSTEM ANALYTICS INTEGRATION       ")
print("=======================================================\n")

# 1. CRITICAL WINDOWS ANACONDA FIX: Force worker worker threads to use the exact active conda python executable
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# Apply Hadoop and Java environment mappings
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ["PATH"]

search_path = r"C:\Program Files\Eclipse Adoptium\jdk-17*"
found_folders = glob.glob(search_path)
if found_folders:
    os.environ["JAVA_HOME"] = found_folders[0]
    os.environ["PATH"] = os.path.join(found_folders[0], "bin") + os.pathsep + os.environ["PATH"]

# 2. Initialize Standalone Spark Session with strict local worker rules
spark = SparkSession.builder \
    .appName("AUCA_Integrated_Analytics") \
    .master("local[*]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.network.timeout", "800s") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("[Pipeline] Ingesting financial transactions and user profiles from MongoDB...")
with open("transactions.json", "r") as f:
    transactions_data = json.load(f)

# Convert raw data array into a Spark DataFrame safely
mongo_transactions_df = spark.createDataFrame(pd.DataFrame(transactions_data[:2000]))

mock_mongo_users = [
    {"user_id": "user_007129", "age_group": "25-34", "country": "Rwanda"},
    {"user_id": "user_000042", "age_group": "18-24", "country": "Kenya"},
    {"user_id": "user_001122", "age_group": "35-44", "country": "Uganda"}
]
mongo_users_df = spark.createDataFrame(pd.DataFrame(mock_mongo_users))

print("[Pipeline] Scanning user streaming session duration metrics from HBase...")
mock_hbase_sessions = [
    {"user_id": "user_007129", "total_session_duration_sec": 4500},
    {"user_id": "user_000042", "total_session_duration_sec": 12800},
    {"user_id": "user_001122", "total_session_duration_sec": 1500}
]
hbase_engagement_df = spark.createDataFrame(pd.DataFrame(mock_hbase_sessions))

print("[Pipeline] Executing cross-database distributed join on 'user_id'...")

# Aggregate transactional records per user profile
user_spend_df = mongo_transactions_df.groupBy("user_id").agg(
    spark_sum("total").alias("lifetime_monetary_spend"),
    spark_count("transaction_id").alias("total_purchase_events")
)

# Join tables
integrated_df = mongo_users_df \
    .join(user_spend_df, on="user_id", how="inner") \
    .join(hbase_engagement_df, on="user_id", how="inner")

# Register View for SQL execution
integrated_df.createOrReplaceTempView("unified_customer_metrics")

clv_dashboard_query = """
    SELECT 
        country                            AS geographic_region,
        age_group                          AS customer_segment,
        COUNT(user_id)                     AS total_active_customers,
        ROUND(AVG(lifetime_monetary_spend), 2) AS avg_lifetime_value,
        ROUND(SUM(total_session_duration_sec) / 60, 1) AS total_engagement_minutes
    FROM unified_customer_metrics
    GROUP BY country, age_group
    ORDER BY avg_lifetime_value DESC
"""

print("\n Integrated Customer Lifetime Value (CLV) Analysis Dashboard:")
spark.sql(clv_dashboard_query).show(truncate=False)

# Terminate session worker threads safely
spark.stop()



      MULTI-SYSTEM ANALYTICS INTEGRATION       

[Pipeline] Ingesting financial transactions and user profiles from MongoDB...
[Pipeline] Scanning user streaming session duration metrics from HBase...
[Pipeline] Executing cross-database distributed join on 'user_id'...

 Integrated Customer Lifetime Value (CLV) Analysis Dashboard:
+-----------------+----------------+----------------------+------------------+------------------------+
|geographic_region|customer_segment|total_active_customers|avg_lifetime_value|total_engagement_minutes|
+-----------------+----------------+----------------------+------------------+------------------------+
|Rwanda           |25-34           |1                     |1340.89           |75.0                    |
|Uganda           |35-44           |1                     |723.38            |25.0                    |
+-----------------+----------------+----------------------+------------------+------------------------+

